# Test: choose an area in Jupyter (`duckosm.AreaPicker`)

This notebook tests the widget on Monaco. Run the cells in sequence.
You must have `pip install "duckosm[widget]"` and a connection to the internet.
The guide is `docs/guides/area-picker.md`.

## 1. Show the widget

Make sure that you see a map with the tabs Search, OSM id, Box, H3 cell and File.
Try each tab. For each choice, make sure that the line under the map shows the extract and its size,
and that the map shows the boundary in blue. Do not click **Get** yet.

In [ ]:
import time
import duckosm

picker = duckosm.AreaPicker(out="data")
picker

## 2. Set the area from Python (a box around Monaco)

The widget runs `duckosm get --dry-run`: it finds the boundary and the extract, and downloads nothing.
The map above moves to Monaco.

In [ ]:
picker.area = {"mode": "bbox", "bbox": [7.42, 43.728, 7.44, 43.738], "name": "monaco_box"}
for _ in range(300):                      # 60 s at most
    if picker.plan:
        break
    time.sleep(0.2)
assert "error" not in picker.plan, picker.plan
x = picker.plan["extract"]
print(picker.plan["name"], "->", x["name"], f"({x['id']}),", round(x["bytes"] / 2**20, 1), "MB,",
      "in the cache" if x["cached"] else "to download")
assert x["id"] == "monaco", x
assert picker.plan["boundary"]["geometry"]["type"] == "Polygon"

## 3. Get the OSM file

Click **Get** in the widget above, or run the next cell. The status line shows the progress.

In [ ]:
area = picker.get()
print(area)
assert area.pbf.is_file() and area.pbf.stat().st_size > 0
assert area.boundary.is_file()
assert picker.result is area

## 4. Wrong input gives an error, not a crash

The line under the map shows the error.

In [ ]:
picker.area = {"mode": "h3", "h3_cell": "not-a-cell"}
for _ in range(300):
    if picker.plan:
        break
    time.sleep(0.2)
print(picker.plan)
assert "error" in picker.plan

## 5. Search and a GeoJSON file

Do these steps in the widget:

1. In the tab **Search**, type `Monaco`, push Enter and click the match with the boundary. Make sure that the boundary of Monaco shows.
2. In the tab **File**, load a GeoJSON file from the computer of the browser. Step 3 wrote `data/monaco_box.geojson`: download it first from the JupyterLab file browser. Make sure that the box shows.
3. Click **Get** and make sure that the status line shows `wrote ...`.

In [ ]:
print(picker.area.get("mode"), picker.plan.get("extract", {}).get("id"), picker.status)

## Build the network

`duckosm build` makes the routable road network of the area in one DuckDB file. It reads the OSM file and the
boundary that **Get** wrote. Monaco takes some seconds.

In [ ]:
import os, re, subprocess, sys, duckdb, ipywidgets as w
from IPython.display import display
area = picker.result or picker.get()   # waits for Get, or runs it
db = area.pbf.with_suffix("").with_suffix(".duckdb")
bar, step = w.IntProgress(min=0, max=100, description="build"), w.Label("starting ...")
display(w.HBox([bar, step]))
build = subprocess.Popen([sys.executable, "-m", "duckosm", "build", "--pbf", str(area.pbf), "--boundary", str(area.boundary),
                          "-o", str(db), "-m", "driving", "-m", "walking"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         env={**os.environ, "FORCE_COLOR": "1"})      # the program draws its progress bar only for a terminal
log, rest = [], ""
progress = re.compile(r"[⠀-⣿]\s+(.+?)\s+[━╸╺]+\s+(\d+)%")     # "⠋ Loading PBF file ━━━╸━━━  12% 0:00:02"
while chunk := os.read(build.stdout.fileno(), 65536):
    text = re.sub(r"\x1b\[[0-9;?]*[A-Za-z]", "", rest + chunk.decode(errors="replace"))   # no color codes
    *parts, rest = re.split(r"[\r\n]", text)
    for part in parts:
        if m := progress.search(part):
            bar.value, step.value = int(m[2]), m[1].strip(" .")
        log.append(part)
if build.wait():
    raise RuntimeError("duckosm build failed:\n" + "\n".join(x for x in log if x.strip() and not progress.search(x))[-3000:])
bar.value, step.value = 100, f"done: {db}"
db

In [ ]:
import duckdb
con = duckdb.connect(str(db), read_only=True)
# the edges per mode: each row is one direction of a road; edge_id is the key that other tools join on
for mode in ("driving", "walking"):
    n, km = con.execute(f"SELECT count(*), round(sum(length_m) / 1000, 1) FROM {mode}.edges").fetchone()
    print(f"{mode}: {n} edges, {km} km")
con.close()

In [ ]:
# the fastest path by car across the area: from a west end to an east end of the main roads (primary to tertiary).
# The boundary cuts some roads, so not each pair of points has a path: the cell tries the 10 nodes at each end.
con = duckdb.connect(str(db), read_only=True)
con.execute("LOAD spatial")
nodes = con.execute("""
    SELECT DISTINCT ST_X(n.geom) AS lon, ST_Y(n.geom) AS lat FROM driving.edges e JOIN driving.nodes n ON n.node_id = e.source
    WHERE e.highway IN ('primary', 'secondary', 'tertiary') ORDER BY lon""").fetchall()
con.close()
r = None
for a in nodes[:10]:
    for b in nodes[::-1][:10]:
        r = duckosm.route_points(str(db), a, b)
        if r:
            break
    if r:
        break
if r:
    print(f"{a} -> {b}: {round(r['time_s'])} s, {round(r['length_m'])} m, {len(r['edges'])} edges")
else:
    print("no path by car between the two ends of the main roads")
